# Functions for DIIM Economic Analysis

This notebook contains the core functions for the Dynamic Inoperability Input-Output Model (DIIM) analysis,
including data loading, model computation, and simulation comparison functions.

## 1. Load Libraries

In [1]:
library(openxlsx)

Warning message:
“package ‘openxlsx’ was built under R version 4.3.3”


## 2. Download COVID Data

Load the 2019 Input-Output Table, extract the technical coefficient matrix `A`, total output `x`,
final demand `c`, sector initial inoperability `q0`, and perturbation `c_star`.
Also computes the interdependency matrix `A*`.

In [2]:
download_data <- function() {
  # download 2019 IOT and extract total output and final demand column
  data <- read.xlsx("dataset/Input_Output_Tables_2019_2021.xlsx", sheet = "2019IOTable")
  iot2019 <- data[2:(2 + 15 - 1), 3:(3 + 15 - 1)]
  x <- data[2:(2 + 15 - 1), ncol(data)]
  c <- data[2:(2 + 15 - 1), ncol(data) - 1]
  x <- as.matrix(sapply(x, as.numeric))
  c <- as.matrix(sapply(c, as.numeric))

  # extract technical coefficient matrix
  A <- read.xlsx("dataset/Input_Output_Tables_2019_2021.xlsx", sheet = "A")
  A <- A[2:(2 + 15 - 1), 3:(3 + 15 - 1)]
  A <- as.matrix(sapply(A, as.numeric))

  # extract sector initial inoperability
  q0_df <- read.xlsx("dataset/covid-data/Unemployment_and_Impact_Analysis.xlsx", sheet = "sector inoperability", colNames = FALSE)
  q0 <- as.numeric(q0_df[2:16, 3])

  c_star_df <- read.xlsx("dataset/covid-data/Unemployment_and_Impact_Analysis.xlsx", sheet = "c_star")
  c_star <- as.numeric(c_star_df[1:15, 7])

  A_star <- solve(diag(as.vector(x))) %*% A %*% diag(as.vector(x))

  return(list(
    x = x,
    c = c,
    A = A,
    q0 = q0,
    c_star = c_star,
    A_star = A_star
  ))
}

## 3. Download Manpower Disruption Data

Load the 2022 Input-Output Table for the manpower disruption scenario.
Extracts `A`, `x`, `q0`, and computes `A*`. Sets `c` and `c_star` to zero
since this scenario has no external demand shock.

In [3]:
download_manpower_data <- function() {
  # Load 2022 IOT for A matrix and x vector
  data_A <- read.xlsx("dataset/manpower-disruption-data/2022_Input_Output_Table.xlsx", sheet = "A", colNames = FALSE)
  num_sectors <- nrow(data_A) - 1
  A <- data_A[2:(num_sectors + 1), 2:(num_sectors + 1)]
  A <- as.matrix(sapply(A, as.numeric))

  # Sheet "x" contains the total output
  data_x <- read.xlsx("dataset/manpower-disruption-data/2022_Input_Output_Table.xlsx", sheet = "x", colNames = FALSE)
  x <- data_x[2:(num_sectors + 1), 2]
  x <- as.numeric(x)

  # Load sector initial inoperability (q0)
  data_q0 <- read.xlsx("dataset/manpower-disruption-data/Sector_initial_inoperability.xlsx", sheet = "Sector_initial_inoperability", colNames = FALSE)
  q0 <- data_q0[2:(num_sectors + 1), 4]
  q0 <- as.numeric(q0)

  # Initialize c and c_star to 0 for manpower disruption scenario
  c <- rep(0, num_sectors)
  c_star <- rep(0, num_sectors)

  # Calculate A_star
  A_star <- solve(diag(x)) %*% A %*% diag(x)

  return(list(
    x = as.matrix(x),
    c = as.matrix(c),
    A = A,
    q0 = q0,
    c_star = c_star,
    A_star = A_star
  ))
}

## 4. DIIM Model

The Dynamic Inoperability Input-Output Model (DIIM) computes the time evolution of sector inoperability
and cumulative economic losses.

**Parameters:**
- `q0`: initial inoperability vector
- `A_star`: interdependency matrix
- `c_star`: demand perturbation vector
- `x`: total output vector
- `lockdown_duration`: days of active shock
- `total_duration`: total simulation days
- `key_sectors`: optional sectors to apply 10% inoperability reduction
- `risk_management`: scaling factor (default 1)
- `days_in_year`: for daily output conversion (default 366)

In [4]:
DIIM <- function(q0, A_star, c_star, x, lockdown_duration, total_duration, key_sectors = NULL, risk_management = 1, days_in_year = 366) {
  a_ii <- diag(A_star)
  T <- total_duration

  num_sectors <- length(q0)
  inoperability_evolution <- matrix(NA, nrow = num_sectors, ncol = total_duration)

  # if key_sectors is not NULL, reduce those sectors' initial q0 by 10%
  if (!is.null(key_sectors)) {
    q0[key_sectors] <- q0[key_sectors] * 0.9
  }

  # we assume after 2 years the economic activity return to 99% of pre lock down level
  q0[q0 == 0] <- 1e-8
  qT = q0 * 1/100

  k <- log(q0 / qT) / (T * (1 - a_ii))
  K <- diag(as.vector(k))

  inoperability_evolution[, 1] <- q0

  for (t in 2:total_duration) {
    if (t <= lockdown_duration) {
      inoperability_evolution[, t] <- inoperability_evolution[, t - 1] + K %*% (A_star %*% inoperability_evolution[, t - 1] + c_star - inoperability_evolution[, t - 1]) * risk_management
    } else {
      # after lockdown c* become 0 as no more external shock
      inoperability_evolution[, t] <- inoperability_evolution[, t - 1] + K %*% (A_star %*% inoperability_evolution[, t - 1] - inoperability_evolution[, t - 1]) * risk_management
    }
  }

  # convert to planned DAILY output (output is in millions)
  x_daily <- x / days_in_year
  EL_evolution <- t(apply(inoperability_evolution, 1, cumsum)) * as.vector(x_daily)
  EL_end <- EL_evolution[, ncol(EL_evolution)]
  total_economic_loss <- sum(EL_evolution[, ncol(EL_evolution)])

  return(
    list(
      inoperability_evolution = inoperability_evolution,
      EL_evolution = EL_evolution,
      EL_end = EL_end,
      total_economic_loss = total_economic_loss
    )
  )
}

## 5. Simulation: ML vs DIIM (COVID Scenario)

Compares total economic loss under three strategies:
1. No intervention (baseline)
2. DIIM-based intervention (top 5 sectors by economic loss)
3. ML-based intervention (top 5 sectors identified by PCA)

In [5]:
simulation_ml_vs_diim <- function(q0, A_star, c_star, x, lockdown_duration, total_duration) {
  # run model to calculate the economic loss (without intervention)
  model <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration)
  EL_evolution <- model$EL_evolution

  # obtain the sectors with top economic loss
  max_econ_loss <- apply(EL_evolution, 1, max)
  sorted_indices <- order(max_econ_loss, decreasing = TRUE)
  top_econ_loss_5 <- sorted_indices[1:5]

  # rerun the model with intervention for the top economic sectors
  model_diim <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration, key_sectors = top_econ_loss_5)

  ml_key_sectors <- c(3, 2, 5, 8, 10) # obtained from PCA
  model_ml <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration, key_sectors = ml_key_sectors)

  model_tot_econ_loss <- model$total_economic_loss
  model_diim_tot_econ_loss <- model_diim$total_economic_loss
  model_ml_tot_econ_loss <- model_ml$total_economic_loss

  return(list(
    lockdown_duration = lockdown_duration,
    total_duration = total_duration,
    model_tot_econ_loss = model_tot_econ_loss,
    model_diim_tot_econ_loss = model_diim_tot_econ_loss,
    model_ml_tot_econ_loss = model_ml_tot_econ_loss
  ))
}

## 6. Simulation: ML vs DIIM (Manpower Disruption Scenario)

Same comparison as above but for the manpower disruption scenario,
using different ML-identified key sectors: `c(79, 107, 89, 11, 33)`.

In [6]:
simulation_ml_vs_diim_manpower <- function(q0, A_star, c_star, x, lockdown_duration, total_duration) {
  # run model to calculate the economic loss (without intervention)
  model <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration)
  EL_evolution <- model$EL_evolution

  # obtain the sectors with top economic loss
  max_econ_loss <- apply(EL_evolution, 1, max)
  sorted_indices <- order(max_econ_loss, decreasing = TRUE)
  top_econ_loss_5 <- sorted_indices[1:5]

  # rerun the model with intervention for the top economic sectors
  model_diim <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration, key_sectors = top_econ_loss_5)

  ml_key_sectors <- c(79, 107, 89, 11, 33) # obtained from PCA
  model_ml <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration, key_sectors = ml_key_sectors)

  model_tot_econ_loss <- model$total_economic_loss
  model_diim_tot_econ_loss <- model_diim$total_economic_loss
  model_ml_tot_econ_loss <- model_ml$total_economic_loss

  return(list(
    lockdown_duration = lockdown_duration,
    total_duration = total_duration,
    model_tot_econ_loss = model_tot_econ_loss,
    model_diim_tot_econ_loss = model_diim_tot_econ_loss,
    model_ml_tot_econ_loss = model_ml_tot_econ_loss
  ))
}

## 7. Quick Test

Run a quick test to verify the functions load and execute correctly.

In [5]:
# Test with COVID data
data <- download_data()
cat("COVID data loaded successfully\n")
cat("Number of sectors:", length(data$q0), "\n")
cat("A matrix dimensions:", dim(data$A), "\n")
cat("Total output (x) sum:", sum(data$x), "\n")

COVID data loaded successfully
Number of sectors: 15 
A matrix dimensions: 15 15 
Total output (x) sum: 1335525 


In [6]:
# Run a short DIIM simulation
result <- DIIM(data$q0, data$A_star, data$c_star, data$x,
               lockdown_duration = 56, total_duration = 730)
cat("DIIM simulation completed\n")
cat("Total economic loss:", result$total_economic_loss, "million\n")

DIIM simulation completed
Total economic loss: 27391.1 million
